In [1]:
import os
import glob
import sqlite3
import pandas as pd
import plotly.express as px

# ==========================================
# 1. LOCATE & LOAD WUENIC EXCEL FILE
# ==========================================
print("--> Searching for WUENIC Excel file...")
excel_files = glob.glob("/kaggle/input/**/wuenic*.xlsx", recursive=True)
if not excel_files:
    excel_files = glob.glob("/kaggle/input/**/*.xlsx", recursive=True)

if not excel_files:
    raise FileNotFoundError("Could not find the WUENIC .xlsx file. Check /kaggle/input.")

xlsx_path = excel_files[0]
print(f"Loading: {xlsx_path}")

# Inspect available sheets (vaccines)
xls = pd.ExcelFile(xlsx_path)
print(f"Discovered {len(xls.sheet_names)} sheets: {xls.sheet_names[:10]}...")

# ==========================================
# 2. MELT & TIDY MULTI-SHEET VACCINE DATA
# ==========================================
all_records = []
ignore_sheets = ["ReadMe", "readme", "regional_global"]

for sheet in xls.sheet_names:
    if any(ig in sheet.lower() for ig in ["readme", "regional"]):
        continue
    
    df_sheet = pd.read_excel(xls, sheet_name=sheet)
    
    # Identify metadata columns vs year columns (e.g. 2000 through 2023)
    id_vars = [c for c in df_sheet.columns if not str(c).strip().isdigit()]
    year_vars = [c for c in df_sheet.columns if str(c).strip().isdigit()]
    
    # Unpivot / Melt wide year columns into long format
    melted = df_sheet.melt(
        id_vars=id_vars,
        value_vars=year_vars,
        var_name="year",
        value_name="coverage_pct"
    )
    
    # Standardize column naming
    melted.columns = [str(c).strip().lower().replace(" ", "_") for c in melted.columns]
    
    # Ensure vaccine identifier exists
    if "vaccine" not in melted.columns:
        melted["vaccine"] = sheet
        
    all_records.append(melted)

df_unified = pd.concat(all_records, ignore_index=True)
df_unified["year"] = pd.to_numeric(df_unified["year"], errors="coerce")
df_unified["coverage_pct"] = pd.to_numeric(df_unified["coverage_pct"], errors="coerce")

print(f"\nUnified Longitudinal Cohort: {df_unified.shape[0]} rows, {df_unified.shape[1]} columns")
print(df_unified.head(4))

# ==========================================
# 3. BUILD EMBEDDED SQL SURVEILLANCE DATABASE
# ==========================================
db_path = "/kaggle/working/wuenic_immunization.db"
conn = sqlite3.connect(db_path)
df_unified.to_sql("national_coverage", conn, if_exists="replace", index=False)
print(f"Loaded records into SQLite at {db_path}")

# ==========================================
# 4. ANALYTICAL SQL QUERIES (SURVEILLANCE)
# ==========================================

# Query 1: Tracking Core Vaccines in Bangladesh over time
sql_bd = """
SELECT 
    country,
    vaccine,
    year,
    coverage_pct
FROM national_coverage
WHERE country = 'Bangladesh' 
  AND vaccine IN ('BCG', 'DTP1', 'DTP3', 'MCV1', 'PCV3', 'ROTAC')
  AND year >= 2005
ORDER BY vaccine, year ASC;
"""
df_bd = pd.read_sql_query(sql_bd, conn)

# Query 2: Post-COVID Immunization Disruption Analysis (2019 vs 2021)
sql_covid_drop = """
SELECT 
    t19.country,
    t19.vaccine,
    t19.coverage_pct AS cov_2019,
    t21.coverage_pct AS cov_2021,
    ROUND(t21.coverage_pct - t19.coverage_pct, 1) AS coverage_drop
FROM national_coverage t19
JOIN national_coverage t21 
  ON t19.country = t21.country 
 AND t19.vaccine = t21.vaccine
WHERE t19.year = 2019 
  AND t21.year = 2021 
  AND t19.vaccine = 'DTP3'
  AND t19.coverage_pct IS NOT NULL 
  AND t21.coverage_pct IS NOT NULL
ORDER BY coverage_drop ASC
LIMIT 10;
"""
df_disruption = pd.read_sql_query(sql_covid_drop, conn)
print("\nTop Countries Facing Greatest DTP3 Immunization Declines (COVID-19 Disruption):")
print(df_disruption.head())

# ==========================================
# 5. INTERACTIVE EPIDEMIOLOGICAL CHARTS
# ==========================================
# Figure 1: Bangladesh Multi-Vaccine Trajectory
fig1 = px.line(
    df_bd,
    x="year",
    y="coverage_pct",
    color="vaccine",
    markers=True,
    title="Bangladesh Pediatric Immunization Coverage Trajectories (2005–2023)",
    labels={"coverage_pct": "Coverage Rate (%)", "year": "Surveillance Year"},
    template="plotly_white"
)
fig1.update_layout(yaxis=dict(range=[0, 105]), hovermode="x unified")
fig1.show()

# Figure 2: South Asia Comparative Benchmarking for DTP3
sql_sa = """
SELECT country, year, coverage_pct
FROM national_coverage
WHERE country IN ('Bangladesh', 'India', 'Pakistan', 'Nepal', 'Sri Lanka')
  AND vaccine = 'DTP3'
  AND year >= 2010;
"""
df_sa = pd.read_sql_query(sql_sa, conn)

fig2 = px.line(
    df_sa,
    x="year",
    y="coverage_pct",
    color="country",
    title="Comparative Pediatric DTP3 Coverage Across South Asia (2010–2023)",
    labels={"coverage_pct": "Coverage (%)", "year": "Year"},
    template="plotly_white"
)
fig2.show()

--> Searching for WUENIC Excel file...
Loading: /kaggle/input/datasets/fahimvj/immunization-data-unicef/wuenic2023rev_web-update.xlsx
Discovered 18 sheets: ['ReadMe', 'BCG', 'DTP1', 'DTP3', 'HEPB3', 'HEPBB', 'HIB3', 'IPV1', 'IPV2', 'MCV1']...

Unified Longitudinal Cohort: 58080 rows, 6 columns
  unicef_region iso3      country vaccine  year  coverage_pct
0          ROSA  AFG  Afghanistan     BCG  2023          68.0
1          ECAR  ALB      Albania     BCG  2023          99.0
2          MENA  DZA      Algeria     BCG  2023          99.0
3          ESAR  AGO       Angola     BCG  2023          73.0
Loaded records into SQLite at /kaggle/working/wuenic_immunization.db

Top Countries Facing Greatest DTP3 Immunization Declines (COVID-19 Disruption):
                                 country vaccine  cov_2019  cov_2021  \
0  Democratic People's Republic of Korea    DTP3      97.0      41.0   
1                                Myanmar    DTP3      90.0      37.0   
2                            